# Model Validation Demo (v0.5.0+)

TerraFlow ships **spatial-block cross-validation** (Roberts et al. 2017) as the
primary validation surface. The Cohen's κ and Moran's I wrappers that lived in
`terraflow.validation` through v0.4.x were removed in v0.5.0 — they didn't earn
Methods-section citations (downstream papers cited `sklearn` / `esda`, not
TerraFlow). This notebook shows:

1. How to invoke spatial-block CV via `terraflow validate`.
2. How to compute Cohen's κ via `sklearn.metrics.cohen_kappa_score` directly on the run's `features.parquet`.
3. How to compute Moran's I via `esda.Moran` on score residuals.

All three patterns are documented in [Migration v0.4 → v0.5](../migration-v0.4-to-v0.5.md).

## 1. Build synthetic inputs + run pipeline

In [1]:
import json, tempfile, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from terraflow.pipeline import run_pipeline
from terraflow.validation import run_validation

tmp = Path(tempfile.mkdtemp())
raster_path = tmp / 'raster.tif'
arr = np.arange(25, dtype='float32').reshape(5, 5)
with rasterio.open(
    raster_path, 'w', driver='GTiff', height=5, width=5, count=1,
    dtype='float32', crs='EPSG:4326',
    transform=from_origin(-100.0, 40.0, 0.01, 0.01),
) as ds:
    ds.write(arr, 1)

climate_path = tmp / 'climate.csv'
pd.DataFrame({
    'lat':        [40.0,  40.01,  40.02],
    'lon':        [-100.0, -99.99, -99.98],
    'mean_temp':  [18.0,   19.0,   20.0],
    'total_rain': [100.0, 120.0,  140.0],
}).to_csv(climate_path, index=False)

config_path = tmp / 'config.yml'
config_path.write_text(textwrap.dedent(f'''
    raster_path: {raster_path}
    climate_csv: {climate_path}
    output_dir: {tmp}/outputs
    roi: {{ type: bbox, xmin: -101.0, ymin: 39.0, xmax: -99.0, ymax: 41.0 }}
    model_params: {{ v_min: 0.0, v_max: 25.0, t_min: 0.0, t_max: 40.0,
                    r_min: 0.0, r_max: 300.0, w_v: 0.4, w_t: 0.3, w_r: 0.3 }}
    max_cells: 50
    validation:
      n_blocks_side: 2
      buffer_deg: 0.01
'''))

df = run_pipeline(config_path)
run_dir = Path(df.attrs['run_dir'])
print('run_dir:', run_dir)

INFO:terraflow:Loaded config from /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/config.yml


INFO:terraflow:Computed run fingerprint kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg (config=7afec49af5b1f5ad3f7c263d3af3747db6d09d9034a5c3794ecc7d7217e0ce16, inputs=2)


INFO:terraflow:DataCatalog built: raster=raster.tif (CRS EPSG:4326, shape (5, 5)), climate=climate.csv (3 rows, vars=['mean_temp', 'total_rain'])


INFO:terraflow:Loaded raster from /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/raster.tif


INFO:terraflow:Loaded climate CSV from /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/climate.csv with 3 rows


...

INFO:terraflow.climate:Linear interpolation failed (QH6154 Qhull precision error: Initial simplex is flat (facet 2 is coplanar with the interior point)

While executing:  | qhull d Qc Qz Qbb Q12 Qt
Options selected for Qhull 2020.2.r 2020/08/31:
  run-id 87473405  delaunay  Qcoplanar-keep  Qz-infinity-point  Qbbound-last
  Q12-allow-wide  Qtriangulate  _pre-merge  _zero-centrum  Qinterior-keep
  Pgood  _max-width 0.02  Error-roundoff 1.4e-13  _one-merge 9.7e-13
  Visible-distance 2.8e-13  U-max-coplanar 2.8e-13  Width-outside 5.5e-13
  _wide-facet 1.7e-12  _maxoutside 1.1e-12

The input to qhull appears to be less than 3 dimensional, or a
computation has overflowed.

Qhull could not construct a clearly convex simplex from points:
- p3(v4):    40 -1e+02 1e+02
- p1(v3):    40 -1e+02   0.1
- p2(v2):    40 -1e+02 -1.9e-14
- p0(v1):    40 -1e+02  0.21

The center point is coplanar with a facet, or a vertex is coplanar
with a neighboring facet.  The maximum round off error for
computing dist

INFO:terraflow:Interpolated climate for 25 cells using strategy='spatial'


INFO:terraflow:Closed raster dataset


INFO:terraflow:Artifacts written to /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/outputs/runs/kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg (fingerprint=kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg, cells=25, total=0.08s)


run_dir: /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/outputs/runs/kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg


## 2. Spatial block cross-validation

Run via the public API. Results land in `report.json` under the `validation` key — block-level scores + buffer-aware fold construction (Roberts et al. 2017).

In [2]:
report_path = run_validation(config_path)
report = json.loads(Path(report_path).read_text(encoding='utf-8'))
print(json.dumps(report.get('validation', {}), indent=2, default=str))

INFO:terraflow:Running validation on /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/outputs/runs/kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg


INFO:terraflow:Validation block written to /var/folders/lw/h_v37nds21q4xf2_q24hhz8h0000gn/T/tmp69w4inen/outputs/runs/kFkTOW74IVyH4q3VVnLtcziyvAbbttiXeSXCOnjKqmg/report.json


{
  "method": "spatial_block_cv",
  "citation": "Roberts et al. 2017, Ecography",
  "n_blocks_side": 2,
  "buffer_deg": 0.01,
  "n_folds": 4,
  "mean_fold_accuracy": 0.8194444444444444,
  "kriging_loocv_rmse": null,
  "note": "model has no free parameters; fold accuracy reflects spatial label consistency, not fit generalization"
}


## 3. Cohen's κ — external (v0.5.0 pattern)

Removed from `terraflow.validation` in v0.5.0. Compute directly via `sklearn.metrics.cohen_kappa_score` against any reference label set. Below: a synthetic reference DataFrame matched to the run's `features.parquet`.

In [3]:
from sklearn.metrics import cohen_kappa_score
features = pd.read_parquet(run_dir / 'features.parquet')
rng = np.random.default_rng(7)
ref_labels = rng.choice(['low','medium','high'], size=len(features))
kappa = cohen_kappa_score(ref_labels, features['label'].to_numpy())
print(f'cohen_kappa (synthetic reference): {kappa:.3f}')

cohen_kappa (synthetic reference): 0.010


## 4. Moran's I — external (v0.5.0 pattern)

Removed from `terraflow.validation` in v0.5.0. Compute spatial autocorrelation directly via `esda.Moran` on score residuals using a k-NN spatial weights matrix.

In [4]:
try:
    import esda, libpysal
    coords = features[['lon','lat']].to_numpy()
    w = libpysal.weights.KNN.from_array(coords, k=min(8, len(features)-1))
    w.transform = 'r'
    mi = esda.Moran(features['score'].to_numpy(), w)
    print(f"Moran's I: {mi.I:.4f} (p_norm={mi.p_norm:.4f})")
except ImportError:
    print('esda + libpysal not installed. Install with: pip install esda libpysal')

esda + libpysal not installed. Install with: pip install esda libpysal


## Why these wrappers were cut

Both Cohen's κ and Moran's I are one-call patterns against `features.parquet`. Maintaining thin wrappers in `terraflow.validation` earned no Methods-section citations (downstream users cite `sklearn` / `esda` directly), so they were removed in v0.5.0 to keep the validation surface focused on **spatial-block CV** — the one diagnostic for which TerraFlow's per-cell schema + run fingerprinting genuinely add value. See issue #136 + PR #142.